# Cognee on TypeDB

End-to-end walkthrough of the TypeDB graph adapter for cognee: start a server, point cognee at it, build a knowledge graph, query it, and look at what landed in TypeDB.

**Prerequisites**

- A TypeDB 3.12+ server. Easiest, from the package directory: `docker compose -f examples/docker/docker-compose.yml up -d --wait` (listens on `127.0.0.1:1730`).
- An LLM key in the environment (`LLM_API_KEY`; see `.env.example` for an Anthropic + local-embeddings setup).
- This package installed: `pip install cognee-community-graph-adapter-typedb` (or `uv sync` in the package directory).

In [ ]:
import os

# Per-dataset TypeDB databases for cognee's backend access control. Cognee reads
# this when it builds its config, so set it before importing cognee.
os.environ.setdefault("GRAPH_DATASET_DATABASE_HANDLER", "typedb")

import cognee

from cognee_community_graph_adapter_typedb import register

register()  # adds the "typedb" graph provider and dataset handler to cognee
cognee.config.set_graph_database_provider("typedb")
cognee.config.set_graph_db_config(
    {
        "graph_database_url": os.environ.get("GRAPH_DATABASE_URL", "127.0.0.1:1730"),
        "graph_database_username": os.environ.get("GRAPH_DATABASE_USERNAME", "admin"),
        "graph_database_password": os.environ.get("GRAPH_DATABASE_PASSWORD", "password"),
        "graph_dataset_database_handler": "typedb",
    }
)

## Ingest and cognify

`add` registers the documents in a dataset; `cognify` extracts entities and relationships with the LLM and writes the graph to a TypeDB database named `cognee_<dataset uuid hex>`.

In [ ]:
documents = [
    "TypeDB is a polymorphic database with a conceptual data model.",
    "TypeQL is TypeDB's declarative query language.",
    "Knowledge graphs represent entities and the relationships between them.",
    "Cognee builds AI memory by combining knowledge graphs with vector search.",
]

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
await cognee.add(documents, "typedb_knowledge")
await cognee.cognify(["typedb_knowledge"])

## Search

In [ ]:
results = await cognee.search(
    query_type=cognee.SearchType.GRAPH_COMPLETION,
    query_text="How does cognee use knowledge graphs?",
    datasets=["typedb_knowledge"],
)
for result in results:
    print(result)

## Look at the graph in TypeDB

The adapter stores cognee's property graph in a reified schema: one `node` entity type and one `edge` relation type, with the payload in `properties-json` and the label/name promoted to attributes. Raw TypeQL goes through the adapter's `query()`; values travel through the `given` stage rather than string interpolation.

In [ ]:
from cognee.context_global_variables import set_database_global_context_variables
from cognee.infrastructure.databases.graph import get_graph_engine
from cognee.modules.data.methods import get_datasets_by_name
from cognee.modules.users.methods import get_default_user

user = await get_default_user()
dataset = (await get_datasets_by_name(["typedb_knowledge"], user.id))[0]
# Under backend access control the graph engine is per dataset: select it
# (the selection persists for the cells below).
async with set_database_global_context_variables(dataset.id, dataset.owner_id):
    graph = await get_graph_engine()
print("database:", graph.database_name)

nodes, edges = await graph.get_graph_data()
print(len(nodes), "nodes,", len(edges), "edges")

rows = await graph.query("match $n isa node, has node-type $t; reduce $count = count groupby $t;")
for row in rows:
    print(f"{row['t']:>20}  {row['count']}")

In [ ]:
rows = await graph.query(
    """given $type: string;
    match $n isa node, has node-type == $type, has name $name;
    select $name; sort $name; limit 10;""",
    {"type": "Entity"},
)
[row["name"] for row in rows]

## Visualize

`visualize_graph` renders the dataset's graph to a standalone HTML file.

In [ ]:
from IPython.display import IFrame

await cognee.visualize_graph("graph.html", dataset="typedb_knowledge")
IFrame("graph.html", width="100%", height=600)